# Matrix-Free Rayleigh Relaxation — full Colab workflow

This notebook keeps the complete workflow for the revised paper:

1. correctness test,
2. harmonic oscillator sanity test,
3. hydrogen atom single-grid test,
4. hydrogen atom \(N\)-sweep and energy-convergence plots,
5. large-\(N\) hydrogen mid-slice heatmap,
6. \( \mathrm{H}_2^+ \) energy curve,
7. \( \mathrm{H}_2^+ \) bond-plane and transverse heatmaps.

Before running, choose **Runtime → Change runtime type → GPU**.


In [ ]:
!nvidia-smi


## 1. Get the repository

Use Option A after the public repository exists. Use Option B for the ZIP bundle.


In [ ]:
# OPTION A: clone the future public repository.
REPO_URL = "https://github.com/mahdi-sasar/matrix-free-rayleigh-relaxation.git"

# !rm -rf /content/matrix-free-rayleigh-relaxation /content/matrix_free_rayleigh_relaxation
# !git clone $REPO_URL /content/matrix-free-rayleigh-relaxation
# %cd /content/matrix-free-rayleigh-relaxation


In [ ]:
# OPTION B: upload the ZIP bundle if the repository is not yet public.
# Uncomment and run this cell.

# from google.colab import files
# import zipfile, shutil
# uploaded = files.upload()
# zip_name = next(iter(uploaded))
# shutil.rmtree('/content/matrix_free_rayleigh_relaxation', ignore_errors=True)
# with zipfile.ZipFile(zip_name) as zf:
#     zf.extractall('/content')
# %cd /content/matrix_free_rayleigh_relaxation


## 2. Install lightweight dependencies

Do not reinstall TensorFlow in Colab unless absolutely necessary. The `--no-deps` flag is intentional.


In [ ]:
!python -m pip install -q -r requirements-colab.txt
!python -m pip install -q -e . --no-deps


## 3. Optional: mount Google Drive

For production runs, mount Drive first and save outputs there.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/mrsr_results


## 4. Configure TensorFlow and plotting defaults


In [ ]:
import json, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import Image, display

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:
        print("Could not set memory growth:", exc)

tf.keras.backend.set_floatx("float64")
print("Default Keras float:", tf.keras.backend.floatx())

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})


## 5. Helper functions

These helpers keep paths robust whether outputs are local or in Google Drive.


In [ ]:
SEARCH_ROOTS = [
    Path("results"),
    Path("/content/drive/MyDrive/mrsr_results"),
    Path("/content/drive/My Drive/mrsr_results"),
]

def latest_csv(named):
    candidates = []
    for root in SEARCH_ROOTS:
        if root.exists():
            candidates.extend(root.glob(f"**/{named}"))
    if not candidates:
        raise FileNotFoundError(f"Could not find {named} under {SEARCH_ROOTS}")
    return max(candidates, key=lambda p: p.stat().st_mtime)

def infer_box_from_metadata(outdir, fallback):
    meta_path = Path(outdir) / "metadata.json"
    if meta_path.exists():
        with open(meta_path, "r") as f:
            meta = json.load(f)
        return float(meta.get("box", fallback))
    return float(fallback)

def pretty_hydrogen_slice(slice_csv, box, out_png, title):
    arr = pd.read_csv(slice_csv, header=None).values
    density = arr**2
    fig, ax = plt.subplots(figsize=(6.3, 5.3), constrained_layout=True)
    extent = [0, box, 0, box]
    im = ax.imshow(density.T, origin="lower", extent=extent, aspect="equal", cmap="YlOrRd")
    if np.nanmax(density) > np.nanmin(density):
        xs = np.linspace(extent[0], extent[1], density.shape[0])
        ys = np.linspace(extent[2], extent[3], density.shape[1])
        X, Y = np.meshgrid(xs, ys, indexing="ij")
        levels = np.linspace(np.nanmin(density), np.nanmax(density), 14)[1:-1]
        ax.contour(X, Y, density, levels=levels, colors="white", linewidths=0.45, alpha=0.72)
    ax.set_xlabel("X (Bohr)")
    ax.set_ylabel("Y (Bohr)")
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(r"$|\psi|^2$")
    fig.savefig(out_png, bbox_inches="tight")
    fig.savefig(Path(out_png).with_suffix(".pdf"), bbox_inches="tight")
    print("Saved:", out_png)
    print("Saved:", Path(out_png).with_suffix(".pdf"))
    plt.show()

def nice_energy_plot(df, out_png, title=r"$\mathrm{H}_2^+$ finite-box potential-energy curve"):
    df = df.sort_values("R_Bohr").copy()
    idx = int(df["total_energy_Ry"].idxmin())
    rbest = float(df.loc[idx, "R_Bohr"])
    ebest = float(df.loc[idx, "total_energy_Ry"])

    fig, ax = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
    ax.plot(df["R_Bohr"], df["electronic_energy_Ry"], marker="o", markersize=5, linewidth=2.0, label="Electronic energy")
    ax.plot(df["R_Bohr"], df["total_energy_Ry"], marker="s", markersize=5, linewidth=2.0, label="Total energy")
    ax.scatter([rbest], [ebest], s=90, zorder=5, label=fr"Lowest sampled total ($R={rbest:.3f}$ Bohr)")
    ax.axvline(rbest, linestyle="--", linewidth=1.0, alpha=0.5)
    ax.set_xlabel("Internuclear separation $R$ (Bohr)")
    ax.set_ylabel("Energy (Ry)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25, linewidth=0.8)
    ax.legend(frameon=True)
    fig.savefig(out_png, bbox_inches="tight")
    fig.savefig(Path(out_png).with_suffix(".pdf"), bbox_inches="tight")
    print("Saved:", out_png)
    print("Saved:", Path(out_png).with_suffix(".pdf"))
    plt.show()
    return idx, rbest, ebest


## 6. Small correctness test


In [ ]:
!pytest -q tests/test_small_dense_comparison.py


## 7. Harmonic oscillator validation

This is the safest first scientific sanity check because the potential is smooth.


In [ ]:
!python examples/run_harmonic_oscillator_3d.py \
    --n 48 \
    --box 12.0 \
    --alpha 0.25 \
    --sigma 0.9 \
    --tol 1e-8 \
    --max-iter 50000 \
    --out results/colab_harmonic_48

pd.read_csv("results/colab_harmonic_48/history.csv").tail()


# Part A — Hydrogen atom tests

The hydrogen atom remains the main single-center benchmark. The cells below include both a single-grid test and the \(N\)-sweep needed for the energy-convergence figure.


## 8. Hydrogen single-grid run


In [ ]:
!python examples/run_hydrogen_box.py \
    --n 64 \
    --box 10.0 \
    --init gaussian \
    --sigma 0.5 \
    --tol 1e-8 \
    --max-iter 100000 \
    --out results/colab_hydrogen_64

with open("results/colab_hydrogen_64/metadata.json") as f:
    h_meta = json.load(f)
h_meta


## 9. Hydrogen single-grid mid-slice heatmap


In [ ]:
h_box = infer_box_from_metadata("results/colab_hydrogen_64", fallback=10.0)
pretty_hydrogen_slice(
    "results/colab_hydrogen_64/psi_mid_z.csv",
    box=h_box,
    out_png="results/colab_hydrogen_64/psi_mid_z_density_pretty.png",
    title="Hydrogen in a box: mid-slice probability density",
)


## 10. Hydrogen \(N\)-sweep: quick run

This generates the data for energy versus number of grid points. The quick values below are for checking the workflow. For the paper, use the production template in the next cell.


In [ ]:
!python scripts/run_scaling_hydrogen.py \
    --n-values 48 64 80 96 112 128 \
    --box 10.0 \
    --tol 1e-8 \
    --sigma 0.5 \
    --max-iter 120000 \
    --check-every 20 \
    --outdir results/hydrogen_n_sweep_quick

hyd_csv = Path("results/hydrogen_n_sweep_quick/hydrogen_scaling.csv")
hyd = pd.read_csv(hyd_csv)
display(hyd)


## 11. Hydrogen \(N\)-sweep: production template

Use this for the paper-quality convergence curve. It saves the largest-\(N\) mid-slice automatically. Uncomment after mounting Google Drive.


In [ ]:
# !python scripts/run_scaling_hydrogen.py \
#     --n-values 64 80 96 112 128 160 192 \
#     --box 10.0 \
#     --tol 1e-8 \
#     --sigma 0.5 \
#     --max-iter 200000 \
#     --check-every 20 \
#     --outdir /content/drive/MyDrive/mrsr_results/hydrogen_n_sweep_box10_production


## 12. Publication plots for Hydrogen energy convergence

This plots:
- final energy versus total grid points,
- absolute approach toward the free-hydrogen value \(-1\) Ry,
- wall time versus total grid points.


In [ ]:
# Use the newest hydrogen sweep CSV, whether local or in Drive.
import subprocess

hyd_csv = latest_csv("hydrogen_scaling.csv")
HYD_OUT = hyd_csv.parent
print("Using hydrogen sweep folder:", HYD_OUT)
print("Using CSV:", hyd_csv)

subprocess.run([
    "python", "scripts/plot_hydrogen_sweep.py",
    str(hyd_csv),
    "--outdir", str(HYD_OUT),
    "--pdf",
], check=True)

display(Image(str(HYD_OUT / "hydrogen_energy_vs_gridpoints.png")))
display(Image(str(HYD_OUT / "hydrogen_energy_error_vs_gridpoints.png")))


## 13. Largest-\(N\) Hydrogen mid-slice heatmap

This automatically finds the largest \(N\) from the sweep and plots its saved mid-slice. This is the figure that should be used in the paper, not the small \(N=64\) diagnostic slice.


In [ ]:
hyd = pd.read_csv(hyd_csv)
hyd = hyd.sort_values("n_per_dim")
largest = hyd.iloc[-1]
N_large = int(largest["n_per_dim"])
slice_dir = largest.get("slice_dir", "")

if not isinstance(slice_dir, str) or not slice_dir:
    # Fallback for older sweep outputs.
    slice_dir = str(HYD_OUT / f"N_{N_large:04d}")

slice_dir = Path(slice_dir)
slice_csv = slice_dir / "psi_mid_z.csv"
print("Largest N:", N_large)
print("Looking for largest-N slice:", slice_csv)

if not slice_csv.exists():
    print("Available hydrogen slices:")
    for p in sorted(HYD_OUT.glob("N_*/psi_mid_z.csv")):
        print(" ", p)
    raise FileNotFoundError(slice_csv)

box_large = infer_box_from_metadata(slice_dir, fallback=10.0)
pretty_hydrogen_slice(
    slice_csv,
    box=box_large,
    out_png=slice_dir / "psi_mid_z_density_pretty.png",
    title=fr"Hydrogen in a box: largest-$N$ mid-slice density ($N={N_large}$)",
)


# Part B — \( \mathrm{H}_2^+ \) molecular ion tests

These cells preserve the improved molecular workflow: energy curve, bond-plane heatmap, and transverse heatmap.


## 14. H\(_2^+\) quick curve run


In [ ]:
!python examples/run_h2plus_curve.py \
    --n 48 \
    --box 16.0 \
    --r-values 0.8 1.0 1.2 1.4 1.6 1.8 2.0 2.2 2.5 3.0 3.5 4.0 \
    --sigma 0.5 \
    --tol 1e-7 \
    --max-iter 80000 \
    --check-every 20 \
    --out results/colab_h2plus_curve_48


## 15. H\(_2^+\) production template

This is the paper-oriented run. Uncomment after mounting Google Drive.


In [ ]:
# !python examples/run_h2plus_curve.py \
#     --n 96 \
#     --box 20.0 \
#     --r-values 0.8 1.0 1.2 1.4 1.6 1.8 2.0 2.2 2.4 2.6 3.0 3.5 4.0 5.0 \
#     --sigma 0.5 \
#     --tol 1e-8 \
#     --max-iter 200000 \
#     --check-every 20 \
#     --out /content/drive/MyDrive/mrsr_results/h2plus_curve_n96_box20


## 16. H\(_2^+\) energy curve plot


In [ ]:
h2_csv = latest_csv("h2plus_curve.csv")
H2_OUT = h2_csv.parent
print("Using H2+ output folder:", H2_OUT)
h2 = pd.read_csv(h2_csv)
display(h2)

idx, R_best, E_best = nice_energy_plot(h2, H2_OUT / "h2plus_curve_pretty.png")
print(f"Lowest sampled total-energy point: R = {R_best:.6f} Bohr, E_total = {E_best:.12f} Ry")


## 17. H\(_2^+\) bond-plane heatmap

This is the main molecular wavefunction figure. It contains the bond axis and marks both protons.


In [ ]:
import subprocess

subprocess.run([
    "python", "scripts/plot_h2plus_slices.py",
    str(H2_OUT),
    "--R", "best",
    "--plane", "bond",
    "--pdf",
], check=True)

# Display the generated PNG.
bond_candidates = sorted(H2_OUT.glob("R_*/h2plus_bond_density_pretty.png"))
if not bond_candidates:
    raise FileNotFoundError("No H2+ bond-plane PNG was generated.")
display(Image(str(max(bond_candidates, key=lambda p: p.stat().st_mtime))))


## 18. H\(_2^+\) transverse mid-slice heatmap

This is a supplementary molecular figure showing the transverse density through the molecular midpoint.


In [ ]:
import subprocess

subprocess.run([
    "python", "scripts/plot_h2plus_slices.py",
    str(H2_OUT),
    "--R", "best",
    "--plane", "transverse",
    "--pdf",
], check=True)

trans_candidates = sorted(H2_OUT.glob("R_*/h2plus_transverse_density_pretty.png"))
if not trans_candidates:
    raise FileNotFoundError("No H2+ transverse PNG was generated.")
display(Image(str(max(trans_candidates, key=lambda p: p.stat().st_mtime))))


## 19. Copy local results to Google Drive

Use this after local quick runs if Drive is mounted.


In [ ]:
# !mkdir -p /content/drive/MyDrive/mrsr_results
# !cp -r results/* /content/drive/MyDrive/mrsr_results/
